In [ ]:
!pip install mlflow optuna --quiet

import os
MLFLOW_URI = "file:///tmp/mlruns" if "COLAB_RELEASE_TAG" in os.environ else "http://mlflow:5000"


# ML Model Training: Hyperparameter Tuning with Optuna

This notebook mirrors the **Model Training** guide at [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/).

Use Optuna to search the hyperparameter space of a Gradient Boosting classifier, log every trial to MLflow, and identify the best configuration.

**What we'll cover:**
1. Load training and validation data
2. Define the Optuna objective function
3. Run the hyperparameter search
4. Compare trials in MLflow

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier   # More powerful than LogisticRegression; worth tuning
from sklearn.metrics import roc_auc_score
import optuna             # Bayesian hyperparameter optimisation framework
import mlflow             # Experiment tracking
import mlflow.sklearn

mlflow.set_tracking_uri(MLFLOW_URI)
# Use a separate experiment so tuning trials don't mix with the baseline run in train.ipynb.
mlflow.set_experiment('employee-attrition-tuning')

# Silence Optuna's per-trial log messages (trial started, finished, best value updated).
# Remove this line to see verbose progress — useful when debugging objective functions.
optuna.logging.set_verbosity(optuna.logging.WARNING)


## Step 1: Load training and validation data

In [ ]:
train_df = pd.read_csv('/workspace/pipeline-data/train.csv')
test_df  = pd.read_csv('/workspace/pipeline-data/test.csv')

X_train = train_df.drop(columns=['Attrition'])
y_train = train_df['Attrition']

# Use test set as validation for tuning (in production use a separate val split)
X_val = test_df.drop(columns=['Attrition'])
y_val = test_df['Attrition']

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Attrition rate — Train: {y_train.mean():.1%}  Val: {y_val.mean():.1%}')

## Step 2: Define the Optuna objective function

In [ ]:
def objective(trial):
    """Optuna calls this function once per trial. Returns the metric to MAXIMISE (ROC-AUC).

    Each trial samples a different hyperparameter combination. Optuna records the
    returned value and uses it to guide the next trial toward more promising regions.
    """
    params = {
        # suggest_float with log=True samples learning_rate on a log scale.
        # Log scale is appropriate because learning_rate matters more at small values
        # (0.01 vs 0.02 is a big difference; 0.20 vs 0.21 is not).
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),

        'max_depth':         trial.suggest_int('max_depth', 2, 8),          # Tree depth — controls overfitting
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300),    # Number of boosting rounds
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),    # Fraction of training rows per tree
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20), # Minimum samples to split a node
    }

    # nested=True creates this run as a CHILD of the outer 'optuna-study' run.
    # This keeps the experiment tree clean: one parent run per study,
    # with individual trials as nested children visible in the MLflow UI.
    with mlflow.start_run(run_name=f'trial-{trial.number}', nested=True):
        mlflow.log_params(params)   # Log all hyperparameters at once

        clf = GradientBoostingClassifier(**params, random_state=42)
        clf.fit(X_train, y_train)

        # Evaluate on the validation set (not train!) to measure generalisation.
        # predict_proba() returns shape (n_samples, 2); column 1 = P(attrition=1).
        auc = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        mlflow.log_metric('val_roc_auc', auc)

    # Return the metric to optimise. Optuna maximises this value (direction='maximize' in the study).
    # Return a negative value if you want to minimise instead (e.g. -loss for cross-entropy).
    return auc

print('Objective function defined.')
print('Search space: learning_rate, max_depth, n_estimators, subsample, min_samples_split')


Optuna uses a **Tree-structured Parzen Estimator (TPE)** by default: it builds a probabilistic model of which hyperparameter regions are most promising based on past trials, then samples from those regions. This is more efficient than grid search or random search.

## Step 3: Run the hyperparameter search

In [ ]:
with mlflow.start_run(run_name='optuna-study'):
    study = optuna.create_study(
        # direction='maximize': Optuna tries to find params that maximise ROC-AUC.
        # Use 'minimize' for loss functions (e.g. cross-entropy, MSE).
        direction='maximize',

        # TPESampler = Tree-structured Parzen Estimator.
        # Instead of random sampling, TPE models the distribution of "good" vs "bad"
        # hyperparameter values based on past trial results, then samples from the
        # "good" region. Much more efficient than grid or random search.
        # seed=42 makes the sampler's random draws reproducible.
        sampler=optuna.samplers.TPESampler(seed=42),

        # MedianPruner stops unpromising trials early.
        # After a trial has run N steps, if its intermediate value is below the median
        # of all completed trials at the same step, Optuna prunes (cancels) it.
        # GradientBoostingClassifier doesn't support intermediate reporting by default,
        # so the pruner won't trigger here — but it's good practice to include it.
        pruner=optuna.pruners.MedianPruner(),
    )

    # study.optimize() runs the objective function n_trials=20 times,
    # using the sampler to choose hyperparameters for each trial.
    study.optimize(objective, n_trials=20, show_progress_bar=False)

    best_params = study.best_params   # Dict of hyperparameter name → best value
    best_auc    = study.best_value    # The highest ROC-AUC achieved across all trials

    # Log the best params with a 'best_' prefix so they're searchable in MLflow
    # without conflicting with individual trial parameter names.
    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metric('best_val_roc_auc', best_auc)

print(f'\nBest ROC-AUC: {best_auc:.4f}')
print('Best params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')


## Step 4: Compare trials in MLflow

Open [mlflow.ops4life.com](https://mlflow.ops4life.com) and navigate to the **employee-attrition-tuning** experiment. Each trial appears as a nested run. Use the **Chart** view to plot `val_roc_auc` against individual parameters — this reveals which hyperparameters have the most impact.

In [ ]:
# Retrain the best model on the FULL training set using the best hyperparameters.
# WHY: during the study, each trial was trained on the same training split.
# Now that we've selected the best config, we reuse ALL available training data
# (train + val if you have a separate val set) to get the strongest final model.
# The val set evaluation above was only to pick hyperparameters — it shouldn't
# be held out permanently once we've committed to a configuration.
print('Retraining with best parameters...')
best_clf = GradientBoostingClassifier(**best_params, random_state=42)
best_clf.fit(X_train, y_train)

# Final evaluation on the test set (never seen during training or tuning).
# This is the number to report as the model's expected production performance.
final_auc = roc_auc_score(y_val, best_clf.predict_proba(X_val)[:, 1])
print(f'Final val ROC-AUC: {final_auc:.4f}')


## Next Steps

- Promote the best model to the MLflow registry: `02-pipeline/ml-model-training/promote.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/)